# Targeted VSR · final experiments (Team Pairadox, IE 643)
Runs everything for the final report on one Colab GPU, in order. Every step is
**resumable**: if Colab disconnects, re-run from the top; finished work is skipped
because it all lives on Drive.

Runtime → Change runtime type → **T4 GPU** (or better).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/IE643_project'   # <- change if you like
import os; os.makedirs(ROOT, exist_ok=True); os.chdir(ROOT)
# First time only: upload tsr_data_baselines.zip to ROOT, then:
if not os.path.exists('tsr'):
    !unzip -q tsr_data_baselines.zip
os.chdir(f'{ROOT}/tsr'); print(os.getcwd())
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
!pip -q install -r requirements.txt gradio
!pip -q install "git+https://github.com/facebookresearch/sam2.git"
!python tests/smoke_test.py | tail -1
!python tests/smoke_exp1_clip.py | tail -1


## Hugging Face login
`stabilityai/stable-video-diffusion-img2vid` is **gated**: open its model page while logged in,
accept the licence, then paste a read token here (or add `HF_TOKEN` to Colab secrets).


In [ ]:
from huggingface_hub import login
login()


## 1 · Data
Experiment 1 clip (copy the mp4 into `data/raw/exp1/`) and DAVIS 2017 480p.


In [ ]:
os.makedirs('data/raw/exp1', exist_ok=True)
# !cp "/content/drive/MyDrive/<where it is>/6655841-hd_1920_1080_30fps.mp4" data/raw/exp1/
if not os.path.exists('data/raw/DAVIS'):
    !wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip -O /content/davis.zip
    !unzip -q /content/davis.zip -d data/raw
!ls data/raw data/raw/exp1


In [ ]:
!python scripts/build_dataset.py --preset folder --name exp1 --root data/raw/exp1 --out data/tubes \
    --targets box_track --boxes-json configs/boxes_exp1.json --max-frames 212 --max-windows 7 --clip-len 27
!python scripts/build_dataset.py --preset davis --root data/raw/DAVIS --out data/tubes \
    --only-splits test --max-clips 8 --max-windows 1 --clip-len 27
!wc -l data/tubes/*.jsonl


## 2 · Baselines and ours (same LR tubes, ×4)
Start with `--limit` to time one sample; `results/<method>/_runtime.jsonl` records
sec/frame and peak VRAM (the numbers for the GPU/memory slide). Remove `--limit` for the full run.


In [ ]:
!python scripts/run_baseline.py --data data/tubes/test --methods bicubic realesrgan --out results
!python scripts/run_baseline.py --data data/tubes/test --methods sd_x4 --out results --limit 2
!python scripts/run_baseline.py --data data/tubes/test --methods ours_svd --out results --limit 2
!tail -n 2 results/*/_runtime.jsonl


In [ ]:
# full runs (long: start these, then work on the report)
!python scripts/run_baseline.py --data data/tubes/test --methods sd_x4 ours_svd --out results


## 3 · Ablations (report: which component matters?)
* strength: how much the prior is allowed to generate (fidelity vs detail)
* no low-frequency fix: does the consistency step help fidelity?
* image prior vs video prior: `sd_x4` (per frame) vs `ours_svd` is the core comparison.
Run on the exp1 tubes only to keep it fast.


In [ ]:
for s in (0.3, 0.7):
    !python scripts/run_baseline.py --data data/tubes/test --filter exp1_ --methods ours_svd --svd-strength {s} --tag _s{s} --out results
!python scripts/run_baseline.py --data data/tubes/test --filter exp1_ --methods ours_svd --svd-fix-sigma 0 --tag _nofix --out results


## 4 · BasicVSR++ / RealBasicVSR
Needs a **separate runtime** (MMagic pins old torch). Open a second Colab notebook, mount the same
Drive folder, and follow the README section. Then come back and evaluate.


## 5 · Evaluate and make figures


In [ ]:
!pip -q install pyiqa
!python scripts/evaluate.py --data data/tubes/test --results results --nr
!python scripts/make_figures.py --data data/tubes/test --results results \
    --methods bicubic realesrgan sd_x4 ours_svd --out figures --limit 8


In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/summary.md').read()))
import glob
for f in sorted(glob.glob('figures/*_frames.png'))[:3]: display(Image(f))
for f in sorted(glob.glob('figures/*_xt.png'))[:2]: display(Image(f))


## 6 · Demo (test interface for the viva and the demo video)


In [ ]:
!python app.py --share
